In [ ]:
!pip install -q GEOparse

import pandas as pd, numpy as np, GEOparse

# If tnbc_data.csv is on your GitHub repo (BioAI-X), you can uncomment the next line to fetch it automatically:
# !wget -q https://raw.githubusercontent.com/<YOUR_GITHUB_USERNAME>/BioAI-X/main/tnbc_data.csv

data = pd.read_csv("tnbc_data.csv", index_col=0)
y = data["label"]
X = data.drop(columns="label")
if X.values.max() > 100:
    X = np.log2(X + 1)

gpl = GEOparse.get_GEO("GPL570", destdir="./").table
m = gpl[["ID", "Gene Symbol"]].dropna()
m = m[~m["Gene Symbol"].str.contains("///")]
mapping = dict(zip(m["ID"], m["Gene Symbol"]))

X = X.T
X["gene"] = X.index.map(mapping)
X = X.dropna(subset=["gene"]).groupby("gene").mean().T
print("Ready! X shape:", X.shape, "| y shape:", y.shape)

FileNotFoundError: [Errno 2] No such file or directory: 'tnbc_data.csv'

In [ ]:
import numpy as np
import pandas as pd
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=42
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

lasso_models = {}
for C in [0.1, 0.3, 1.0]:
  m = Pipeline([
      ("sc", StandardScaler()),
      ("sel", SelectKBest(f_classif, k=500)),
      (
          "clf",
          LogisticRegression(
              penalty="l1",
              solver="liblinear",
              C=C,
              class_weight="balanced",
              max_iter=1000,
              random_state=42,
          ),
      ),
  ])
  auc = cross_val_score(m, X_train, y_train, cv=cv, scoring="roc_auc").mean()
  m.fit(X_train, y_train)
  nz = int((m.named_steps["clf"].coef_ != 0).sum())
  lasso_models[C] = m
  print(f"C={C:<4} | CV AUC={auc:.3f} | non-zero genes={nz}")

In [ ]:
chosen_C = 0.3  # change to 0.1 or 1.0 if another C lands closer to 15-40 genes
best_lasso = lasso_models[chosen_C]

sel_mask = best_lasso.named_steps["sel"].get_support()
selected_genes = np.asarray(X.columns)[sel_mask]
coefs = best_lasso.named_steps["clf"].coef_.ravel()

mean_diff = (
    X.loc[np.asarray(y) == 1, selected_genes].mean(axis=0).values
    - X.loc[np.asarray(y) == 0, selected_genes].mean(axis=0).values
)

lasso_df = pd.DataFrame({
    "Gene": selected_genes,
    "LASSO_Coef": coefs,
    "Abs_Coef": np.abs(coefs),
    "TNBC_minus_NonTNBC": mean_diff,
})

nonzero_df = (
    lasso_df[lasso_df["Abs_Coef"] > 1e-12]
    .sort_values(by="Abs_Coef", ascending=False)
    .reset_index(drop=True)
)

top20_df = nonzero_df.head(20).copy()
top20_df["Direction"] = np.where(
    top20_df["TNBC_minus_NonTNBC"] > 0,
    "Up in TNBC (+)",
    "Down in TNBC (-)",
)

top20_df.to_csv("top20_biomarkers.csv", index=False)
print(f"Saved top20_biomarkers.csv with C={chosen_C} ({len(nonzero_df)} non-zero genes total)")
display(top20_df)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = Pipeline([
    ("sel", SelectKBest(f_classif, k=500)),
    (
        "clf",
        RandomForestClassifier(
            n_estimators=300,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1,
        ),
    ),
])

rf.fit(X_train, y_train)

# Map feature_importances_ back through the SelectKBest mask
rf_genes = X.columns[rf.named_steps["sel"].get_support()]
imp = pd.Series(rf.named_steps["clf"].feature_importances_, index=rf_genes)
rf_top20 = imp.sort_values(ascending=False).head(20)

print("Random Forest Top 20 Genes:")
print(rf_top20)

rf_top20_df = rf_top20.reset_index()
rf_top20_df.columns = ["Gene", "RF_Importance"]
rf_top20_df.to_csv("rf_top20.csv", index=False)

# Check overlap between tuned LASSO top 20 and RF top 20
overlap = sorted(set(top20_df["Gene"]).intersection(set(rf_top20.index)))
print(f"\nOverlapping genes ({len(overlap)}): {overlap}")

In [ ]:
import matplotlib.pyplot as plt

# Use overlapping genes first if available, otherwise use rf_top20.index[:5]
top5 = (
    overlap[:5]
    if len(overlap) >= 5
    else list(dict.fromkeys(list(overlap) + list(rf_top20.index)))[:5]
)

fig, axes = plt.subplots(1, 5, figsize=(15, 3.5), dpi=200)
for ax, g in zip(axes, top5):
  ax.boxplot([X.loc[y == 0, g], X.loc[y == 1, g]], tick_labels=["non-TNBC", "TNBC"])
  ax.set_title(g)
  ax.set_ylabel("Log2 Expression")

plt.tight_layout()
plt.savefig("top5_boxplots.png", dpi=200)
plt.show()
print("Saved top5_boxplots.png for:", top5)

In [ ]:
# Prioritise overlapping genes first, then fill up to 15 from rf_top20
candidate_genes = list(dict.fromkeys(list(overlap) + list(rf_top20.index)))[:15]

summary_rows = []
for g in candidate_genes:
  mean_tnbc = X.loc[y == 1, g].mean()
  mean_non = X.loc[y == 0, g].mean()
  direction = "Up-regulated (+)" if mean_tnbc > mean_non else "Down-regulated (-)"
  summary_rows.append({"Gene": g, "Direction_in_Data": direction})

lit_template_df = pd.DataFrame(summary_rows)
display(lit_template_df)


### Literature Validation (`GSE76275` Top Candidate Biomarkers)

| Gene | Direction in Data | Evidence from Literature | PMID |
|---|---|---|---|
| `AX746627` | Down-regulated (-) | no published TNBC link found. | N/A |
| `C5orf49` | Down-regulated (-) | no published TNBC link found. | N/A |
| `COL18A1-AS1` | Down-regulated (-) | no published TNBC link found. | N/A |
| `FZD9` | Up-regulated (+) | Reported as a prognostic biomarker candidate in an in silico TNBC study. | `33796830` |
| `LOC338667` | Down-regulated (-) | no published TNBC link found. | N/A |
| `RP11-55K13.1` | Down-regulated (-) | no published TNBC link found. | N/A |
| `SKA1` | Up-regulated (+) | A TNBC study reported the opposite direction (discordant with our data). | N/A |
| `GRIK3` | Down-regulated (-) | no directly verified gene-specific TNBC expression evidence found. | N/A |
| `LOC643085` | Down-regulated (-) | no published TNBC link found. | N/A |
| `RP11-400N9.1` | Down-regulated (-) | no published TNBC link found. | N/A |
| `LINC00669` | Up-regulated (+) | no published TNBC link found. | N/A |
| `DEFB132` | Down-regulated (-) | no directly verified gene-specific TNBC expression evidence found. | N/A |
| `TNNI2` | Up-regulated (+) | Previously cited PMID did not match the claimed gene. | N/A |
| `DLX5` | Up-regulated (+) | Previously cited paper concerned DLX4 rather than DLX5. | N/A |
| `BC043356` | Down-regulated (-) | no published TNBC link found. | N/A |

### Known Breast Cancer Marker Sanity Check

The following genes will be examined separately to determine whether the differential expression analysis recovers established breast cancer subtype patterns.

| Gene | Expected Biological Pattern | Direction in Our Data | Literature Evidence | PMID |
|---|---|---|---|---|
| `ESR1` | Often lower in TNBC | Pending analysis | Pending verification | N/A |
| `PGR` | Often lower in TNBC | Pending analysis | Pending verification | N/A |
| `GATA3` | Often lower in basal/TNBC | Pending analysis | Pending verification | N/A |
| `FOXA1` | Often lower in basal/TNBC | Pending analysis | Pending verification | N/A |
| `ERBB2` | Often lower in TNBC | Pending analysis | Pending verification | N/A |
| `KRT5` | Often higher in basal/TNBC | Pending analysis | Pending verification | N/A |
| `KRT17` | Often higher in basal/TNBC | Pending analysis | Pending verification | N/A |

**Note:** Expected biological patterns are reference expectations, not assumed results. Actual expression directions and adjusted p-values must be checked against the GSE76275 results before making claims.

### Limitations

- Some candidate genes have limited or no directly verified TNBC-specific literature evidence.
- SKA1 showed a discordant expression direction compared with a published TNBC gene-signature study.
- Several microarray transcripts are unannotated or represent non-coding RNAs.
- Independent-cohort validation is required to establish the reproducibility of the identified biomarkers.

**Validation Scope:** Validation was done on a held-out test set and literature evidence; independent-cohort validation is future work.


In [ ]:
import pandas as pd

# 1. Check canonical TNBC markers in Day 2 top500_DEGs.csv
deg500 = pd.read_csv("top500_DEGs.csv")
# Adjust column name if your gene column is named 'gene' or 'index' instead of 'Gene'
gene_col = [c for c in deg500.columns if c.lower() in ["gene", "symbol", "unnamed: 0", "index"]][0]
canonical = ["ESR1", "GATA3", "FOXA1", "PGR", "ERBB2", "BCL11A", "KRT5", "KRT14"]

found_canonical = deg500[deg500[gene_col].isin(canonical)]
print("Canonical TNBC markers confirmed in Day 2 Top 500 DEGs:")
display(found_canonical)

# 2. Separate named protein-coding genes from unannotated/ncRNA transcripts in your ML candidate pool
all_ml_genes = list(dict.fromkeys(list(top20_df["Gene"]) + list(rf_top20.index)))
unannotated_prefixes = ("LOC", "RP11-", "RP1-", "RP3-", "RP4-", "RP5-", "AX7", "BC0", "LINC", "AC0", "AL0", "AP0")

named_coding = [
    g for g in all_ml_genes
    if not g.startswith(unannotated_prefixes) and "-AS" not in g and "orf" not in g
]
obscure_transcripts = [g for g in all_ml_genes if g not in named_coding]

print(f"\nNamed protein-coding ML candidates ({len(named_coding)}):", named_coding)
print(f"Unannotated / ncRNA transcripts ({len(obscure_transcripts)}):", obscure_transcripts)

### Known Breast Cancer Marker Sanity Check

| Gene | Dataset direction | FDR |
|---|---|---:|
| ESR1 | Down | 1.688440e-17 |
| PGR | Down | 3.185310e-15 |
| GATA3 | Down | 5.569195e-24 |
| FOXA1 | Down | 5.523808e-20 |
| ERBB2 | Down | 1.305983e-06 |
| KRT5 | Up | 9.072937e-12 |
| KRT17 | Not found | N/A |

**KRT17 note:** KRT17 was not detected under its exact gene symbol. KRT17P5 was identified, but it is a pseudogene and was not substituted for KRT17.

In [1]:
import pandas as pd

data = pd.read_csv("tnbc_data.csv", index_col=0)

y = data["label"].astype(int)
X = data.drop(columns=["label"])

print(X.shape)
print(y.shape)
print(y.value_counts())

(265, 54675)
(265,)
label
1    198
0     67
Name: count, dtype: int64


In [2]:
print(data.columns[:10].tolist())
print(data.index[:5].tolist())

['1007_s_at', '1053_at', '117_at', '121_at', '1255_g_at', '1294_at', '1316_at', '1320_at', '1405_i_at', '1431_at']
['GSM1974566', 'GSM1974567', 'GSM1974568', 'GSM1974569', 'GSM1974570']


In [3]:
gene_expr = pd.read_csv("gene_expression_matrix.csv", index_col=0)

print(gene_expr.shape)
print(gene_expr.head())

(21655, 265)
             GSM1974566  GSM1974567  GSM1974568  GSM1974569  GSM1974570  \
Gene Symbol                                                               
A1BG           5.855517    5.984260    5.929213    5.998468    6.232520   
A1BG-AS1       7.859217    8.334229    7.804293    7.615187    7.792255   
A1CF           5.781842    5.625170    5.594986    5.535535    5.783453   
A2M            9.347806    8.643112    9.624497    9.575210    9.608912   
A2M-AS1        6.806182    6.509811    7.798391    6.701965    6.660856   

             GSM1974571  GSM1974572  GSM1974573  GSM1974574  GSM1974575  ...  \
Gene Symbol                                                              ...   
A1BG           5.913146    6.622700    5.848213    6.185615    6.457763  ...   
A1BG-AS1       7.665171    7.841622    7.617630    7.801114    8.106000  ...   
A1CF           5.557206    5.899482    5.691803    5.699546    5.767611  ...   
A2M            9.354159    9.701424    9.592941    9.124076  

In [4]:
# Transpose: samples become rows, genes become columns
expr = gene_expr.T.copy()

# Match the labels to the sample IDs
labels = y.reindex(expr.index)

print("Missing labels:", labels.isna().sum())

# Combine expression data and labels
final_data = expr.copy()
final_data["label"] = labels.astype(int)

# Save the final dataset
final_data.to_csv("tnbc_data.csv")

print("Final dataset shape:", final_data.shape)
print(final_data["label"].value_counts())

Missing labels: 0
Final dataset shape: (265, 21656)
label
1    198
0     67
Name: count, dtype: int64


In [5]:
data = final_data.copy()

y = data["label"].astype(int)
X = data.drop(columns=["label"])

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (265, 21655)
y shape: (265,)


In [6]:
from google.colab import files
files.download("tnbc_data.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>